# Two-Intersection BML Model

A BML-inspired cellular automaton on an explicit **road network** with exactly
two signalised intersections.

### Road layout

```
        v_col1        v_col2
          |              |
──────────I₁─────────────I₂──────────   ← h_row (horizontal road)
          |              |
```

- **Horizontal road**: one periodic row, cars move right.
- **Vertical road 1 / 2**: two periodic columns, cars move up.
- **Intersection 1** at `(h_row, v_col1)`, **Intersection 2** at `(h_row, v_col2)`.

### Traffic lights

Each intersection has an **independent** phase:

| Phase | I₁ state | I₂ state (with offset) |
|---|---|---|
| 0 … T_green−1 | H→ green · V↑ red | depends on `phase_offset` |
| T_green … T_cycle−1 | V↑ green · H→ red | depends on `phase_offset` |

### BML-style update rule

1. All horizontal cars that can advance (next cell empty **and** local light is
   green for horizontal) move simultaneously.
2. All vertical cars on each column that can advance move simultaneously,
   checking the **already-updated** horizontal road at the intersection and
   the local light.

Cars can never enter an occupied cell; the local traffic light adds an extra
block on intersection entry when the direction has red.

In [ ]:
# ── 1. Imports & Global Parameters ──────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

L         = 80    # grid size (road length = L cells, periodic)
T_CYCLE   = 20    # traffic-light cycle length (steps)
T_WARMUP  = 300   # transient steps discarded before measurement
T_MEASURE = 300   # steps averaged for flow

print(f'L={L}, T_cycle={T_CYCLE}, T_warmup={T_WARMUP}, T_measure={T_MEASURE}')

In [ ]:
# ── 2. TwoIntersectionBML — road network with 2 signalised junctions ────────

class TwoIntersectionBML:
    """
    BML-style cellular automaton with exactly two explicit signalised
    intersections on a road network.

    Road network on an L×L grid:
      - Horizontal road : row h = L//2,  all columns, periodic
      - Vertical road 1 : col v1 = L//3, all rows,    periodic
      - Vertical road 2 : col v2 = 2*L//3, all rows,  periodic
      - Intersection 1 at (h, v1),  Intersection 2 at (h, v2)

    Grid values:
      -1 = non-road   0 = empty road   1 = horizontal car   2 = vertical car

    Traffic lights (per intersection, independent):
      phase_i = (t + offset_i) % T_cycle
      horizontal green  <=>  phase_i < T_cycle // 2
      vertical   green  <=>  phase_i >= T_cycle // 2
    `phase_offset` shifts Intersection 2 relative to Intersection 1.

    Update order (BML-style simultaneous within each direction):
      1. Move all horizontal cars (checking local light at each intersection).
      2. Move all vertical cars on v_col1, then v_col2 (checking updated
         h_row and local light at each intersection).
    """

    def __init__(self, L, density, T_cycle=20, phase_offset=0, seed=None):
        self.L            = L
        self.T_cycle      = T_cycle
        self.phase_offset = phase_offset
        self.t            = 0

        self.h_row  = L // 2
        self.v_col1 = L // 3
        self.v_col2 = 2 * L // 3

        # Build road network: -1 = non-road, 0 = empty road
        self.grid = np.full((L, L), -1, dtype=np.int8)
        self.grid[self.h_row, :]  = 0   # horizontal road
        self.grid[:, self.v_col1] = 0   # vertical road 1
        self.grid[:, self.v_col2] = 0   # vertical road 2

        # Populate cars (intersections left empty at initialisation)
        rng = np.random.default_rng(seed)

        h_cols = np.array([c for c in range(L) if c not in (self.v_col1, self.v_col2)])
        h_mask = rng.random(len(h_cols)) < density
        self.grid[self.h_row, h_cols[h_mask]] = 1

        v_rows = np.array([r for r in range(L) if r != self.h_row])
        for v_col in (self.v_col1, self.v_col2):
            v_mask = rng.random(len(v_rows)) < density
            self.grid[v_rows[v_mask], v_col] = 2

    # ── Light helpers ────────────────────────────────────────────────────────

    def _h_green(self, intersection_idx):
        """True when horizontal has green at intersection 0 or 1."""
        offset = 0 if intersection_idx == 0 else self.phase_offset
        return (self.t + offset) % self.T_cycle < self.T_cycle // 2

    def phase_label(self, i):
        return 'H\u2192' if self._h_green(i) else 'V\u2191'

    # ── Simulation step ──────────────────────────────────────────────────────

    def step(self):
        """
        One full timestep.  Returns the fraction of cars that moved.
        """
        h_green_1 = self._h_green(0)
        h_green_2 = self._h_green(1)
        n_moved   = 0

        # 1. Horizontal movement ─────────────────────────────────────────────
        h     = self.grid[self.h_row, :]
        ahead = np.roll(h, -1)
        can   = (h == 1) & (ahead == 0)
        # Block entry into intersection when local light is red for horizontal
        if not h_green_1:
            can[(self.v_col1 - 1) % self.L] = False
        if not h_green_2:
            can[(self.v_col2 - 1) % self.L] = False
        n_moved += int(can.sum())
        if can.any():
            h_new             = h.copy()
            h_new[can]        = 0
            h_new[np.roll(can, 1)] = 1
            self.grid[self.h_row, :] = h_new

        # 2. Vertical movement (each column checks updated h_row) ─────────────
        for v_col, h_green in ((self.v_col1, h_green_1), (self.v_col2, h_green_2)):
            v    = self.grid[:, v_col]
            abov = np.roll(v, 1)              # abov[i] = v[i-1] (cell above)
            can  = (v == 2) & (abov == 0)
            # Block entry into intersection when local light is red for vertical
            if h_green:
                can[(self.h_row + 1) % self.L] = False
            n_moved += int(can.sum())
            if can.any():
                v_new                  = v.copy()
                v_new[can]             = 0
                v_new[np.roll(can, -1)] = 2   # place one row up
                self.grid[:, v_col]    = v_new

        self.t += 1
        total = int((self.grid > 0).sum())
        return n_moved / max(total, 1)

    # ── Measurement helpers ──────────────────────────────────────────────────

    def warmup(self, T=None):
        for _ in range(T or T_WARMUP):
            self.step()

    def run(self, T=None):
        """Run T measurement steps; return mean flow fraction."""
        return float(np.mean([self.step() for _ in range(T or T_MEASURE)]))

    def h_queue_length(self):
        """
        Mean queue length on the horizontal road: number of consecutive
        horizontal cars immediately before each intersection (both combined).
        """
        h = self.grid[self.h_row, :]
        total_queue = 0
        for v_col in (self.v_col1, self.v_col2):
            q = 0
            pos = (v_col - 1) % self.L
            while h[pos] == 1:
                q   += 1
                pos  = (pos - 1) % self.L
                if q > self.L:
                    break
            total_queue += q
        return total_queue / 2   # average per intersection


# Quick sanity check
s = TwoIntersectionBML(L=L, density=0.3, T_cycle=T_CYCLE, seed=0)
s.warmup(50)
f = s.run(50)
print(f'Sanity check — flow at \u03c1=0.30: {f:.4f}')
print(f'Intersections: I\u2081=({s.h_row},{s.v_col1})  I\u2082=({s.h_row},{s.v_col2})')

## Section 3 — Fundamental Diagram

Flow (fraction of cars moving per step) vs road density ρ for two light
coordination strategies:
- **Synchronized** (`phase_offset = 0`): both intersections switch together.
- **Green-wave** (`phase_offset = T_cycle/2`): Intersection 2 is offset by
  half a cycle so cars released by I₁ arrive at I₂ on green.

In [ ]:
densities  = np.linspace(0.05, 0.95, 28)
flows_sync = []
flows_wave = []

for i, rho in enumerate(densities):
    for offset, collector in [(0, flows_sync), (T_CYCLE // 2, flows_wave)]:
        s = TwoIntersectionBML(L=L, density=rho, T_cycle=T_CYCLE,
                               phase_offset=offset, seed=i)
        s.warmup()
        collector.append(s.run())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(densities, flows_sync, 'o-', color='steelblue', lw=2, ms=5,
        label=f'Synchronized (offset = 0)')
ax.plot(densities, flows_wave, 's-', color='darkorange', lw=2, ms=5,
        label=f'Green-wave (offset = {T_CYCLE // 2})')
ax.set_xlabel('Road density \u03c1')
ax.set_ylabel('Mean flow (fraction of cars moving / step)')
ax.set_title(f'Fundamental Diagram — Two-Intersection BML  (L={L}, T_cycle={T_CYCLE})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

peak_sync = float(densities[np.argmax(flows_sync)])
peak_wave = float(densities[np.argmax(flows_wave)])
print(f'Peak-flow density — Sync: {peak_sync:.2f},  Green-wave: {peak_wave:.2f}')

## Section 4 — Queue Length Analysis

Mean queue length (consecutive horizontal cars stacked before an intersection)
vs density for both coordination modes. Long queues between the two
intersections at high density can **spill back** and block the other intersection.

In [ ]:
q_sync = []
q_wave = []

for i, rho in enumerate(densities):
    for offset, collector in [(0, q_sync), (T_CYCLE // 2, q_wave)]:
        s = TwoIntersectionBML(L=L, density=rho, T_cycle=T_CYCLE,
                               phase_offset=offset, seed=i)
        s.warmup()
        q_vals = [s.h_queue_length() for _ in range(T_MEASURE) if not s.step()]
        # step() and then sample queue each step
        s2 = TwoIntersectionBML(L=L, density=rho, T_cycle=T_CYCLE,
                                phase_offset=offset, seed=i)
        s2.warmup()
        q_meas = []
        for _ in range(T_MEASURE):
            s2.step()
            q_meas.append(s2.h_queue_length())
        collector.append(np.mean(q_meas))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(densities, q_sync, 'o-', color='steelblue', lw=2, ms=5,
        label='Synchronized')
ax.plot(densities, q_wave, 's-', color='darkorange', lw=2, ms=5,
        label='Green-wave')
ax.set_xlabel('Road density \u03c1')
ax.set_ylabel('Mean queue length (cells)')
ax.set_title(f'Queue Length at Intersections  (L={L}, T_cycle={T_CYCLE})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Section 5 — Phase Offset Sweep

At a fixed moderate density (near the flow peak), sweep `phase_offset`
from 0 to `T_cycle − 1` steps and measure mean flow.

The optimal offset is where cars released by I₁ arrive at I₂ just as
it turns green — the **green-wave** condition.  For equally-spaced
intersections and uniform car speed, theory predicts the optimum at
`offset = travel_time mod T_cycle`.

In [ ]:
RHO_CRIT = float(densities[np.argmax(flows_wave)])   # density of peak flow
offsets  = np.arange(0, T_CYCLE)
N_SEEDS  = 5

flows_vs_offset = []
for offset in offsets:
    run_flows = []
    for seed in range(N_SEEDS):
        s = TwoIntersectionBML(L=L, density=RHO_CRIT, T_cycle=T_CYCLE,
                               phase_offset=int(offset), seed=seed)
        s.warmup()
        run_flows.append(s.run())
    flows_vs_offset.append(np.mean(run_flows))

flows_vs_offset = np.array(flows_vs_offset)
best_offset     = int(offsets[np.argmax(flows_vs_offset)])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(offsets, flows_vs_offset, 'o-', color='steelblue', lw=2, ms=6)
ax.axvline(best_offset, color='tomato', lw=1.5, linestyle='--',
           label=f'Best offset = {best_offset} steps')
ax.axvline(T_CYCLE // 2, color='grey', lw=1.2, linestyle=':',
           label=f'T_cycle/2 = {T_CYCLE // 2} steps')
ax.set_xlabel('Phase offset at I\u2082 (steps)')
ax.set_ylabel('Mean flow (fraction of cars moving / step)')
ax.set_title(
    f'Flow vs Phase Offset  (\u03c1 = {RHO_CRIT:.2f}, T_cycle = {T_CYCLE}, '
    f'averaged over {N_SEEDS} seeds)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Optimal phase offset: {best_offset} steps  '
      f'(flow = {flows_vs_offset[best_offset]:.4f})')
print(f'Flow at offset=0 (sync): {flows_vs_offset[0]:.4f}')

## Section 6 — Space-Time Diagram: Horizontal Road

Rows = time steps, columns = horizontal road position.  **Black** = occupied,
**white** = empty.  Gold dashed lines mark the two intersection columns.

Two sub-panels compare synchronized and green-wave offset at the same density.

In [ ]:
N_ST     = 300
RHO_ST   = RHO_CRIT

def make_st(offset, seed=99):
    s = TwoIntersectionBML(L=L, density=RHO_ST, T_cycle=T_CYCLE,
                           phase_offset=offset, seed=seed)
    s.warmup()
    st = np.zeros((N_ST, L), dtype=np.uint8)
    for k in range(N_ST):
        s.step()
        st[k, :] = (s.grid[s.h_row, :] > 0).astype(np.uint8)
    return st, s

st_sync, s_sync = make_st(0)
st_wave, s_wave = make_st(T_CYCLE // 2)

fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, 5),
                                   sharex=True, sharey=True)
fig.suptitle(
    f'Space\u2013Time Diagram: Horizontal Road  '
    f'(\u03c1 = {RHO_ST:.2f}, T_cycle = {T_CYCLE})',
    fontsize=12)

for ax, st_data, label in [
        (ax_l, st_sync, 'Synchronized (offset = 0)'),
        (ax_r, st_wave, f'Green-wave (offset = {T_CYCLE // 2})')]:
    ax.imshow(st_data, cmap='binary', interpolation='nearest',
              aspect='auto', origin='upper')
    ax.set_xlabel('Horizontal road position (column)')
    ax.set_title(label, fontsize=11)
    for v_col, lbl in [(s_sync.v_col1, 'I\u2081'), (s_sync.v_col2, 'I\u2082')]:
        ax.axvline(v_col, color='gold', lw=1.5, linestyle='--')
        ax.text(v_col + 1, N_ST * 0.02, lbl, color='gold',
                fontsize=9, va='top', fontweight='bold')

ax_l.set_ylabel('Time step')
plt.tight_layout()
plt.show()

## Section 7 — Summary

Overlaid fundamental diagrams and a bar chart showing peak flow for both
coordination strategies.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1 — overlaid FD
ax1.plot(densities, flows_sync, 'o-', color='steelblue', lw=2, ms=4,
         label='Synchronized')
ax1.plot(densities, flows_wave, 's-', color='darkorange', lw=2, ms=4,
         label='Green-wave')
ax1.set_xlabel('Road density \u03c1')
ax1.set_ylabel('Mean flow')
ax1.set_title('Fundamental Diagram')
ax1.legend()
ax1.grid(alpha=0.3)

# Panel 2 — bar chart of peak flow and density at peak
labels    = ['Synchronized', 'Green-wave']
peak_f    = [max(flows_sync), max(flows_wave)]
colors    = ['steelblue', 'darkorange']
bars      = ax2.bar(labels, peak_f, color=colors, edgecolor='black', width=0.5)
for bar, val in zip(bars, peak_f):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax2.set_ylabel('Peak flow')
ax2.set_title('Peak Flow Comparison')
ax2.set_ylim(0, max(peak_f) * 1.15)
ax2.grid(axis='y', alpha=0.3)

fig.suptitle(
    f'Two-Intersection BML Summary  (L={L}, T_cycle={T_CYCLE})',
    fontsize=13)
plt.tight_layout()
plt.show()

gain = (max(flows_wave) - max(flows_sync)) / max(flows_sync) * 100
print(f'Green-wave peak flow gain over Synchronized: {gain:+.1f}%')
print(f'Optimal phase offset at critical density:    {best_offset} steps')